Following the [video tutorial](https://www.youtube.com/watch?v=VMj-3S1tku0) from Karpathy.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
def f(x):
    return 3 * x**2 - 4 * x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

(here he explains the meaning of derivatives, taking into account functions with respect to multiple variables. Calculus--I'm so glad I studied physics)

In [42]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        # This is the derivative of L (the end result) with respect to L. At initialization we assume that it has no
        # impact on it.
        self.grad = 0.0
        self._backward = lambda: None
        # this is a set for efficiency reasons, according to Andrej
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, {self.label})"  #  , _prev={self._prev})"

    def __add__(self, other):
        # cover the cases in which we get passed a float or an int
        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # DUDA: dentro de este closure, cuando llamemos a esta función en otro lugar, ese self será el self del
            # valor original que está "creando" un nuevo Value, o el self del Value que está siendo creado?
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward

        return out

    def __radd__(self, other):  # other + self
        # this method gets called when a NotImplemented error gets raised while calling (4).__add__(some_value)
        # (in that case, the int class doesn't know shit about our Value class, so when you try to operate with another
        # class it raises that NotImplemented error)
        return self + other

    def __neg__(self):  # -self
        return self * -1

    def __sub__(self, other):  # self - other
        return self + (-other)

    def __mul__(self, other):
        # cover the cases in which we get passed a float or an int
        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


a = Value(2.0, label="a")
b = Value(3.4, label="b")
c = Value(-0.4, label="c")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
f = Value(-2.9, label="f")
L = d * f
L.label = "L"

In [3]:
from graphviz import Digraph


def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        # dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape="record")
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

This is what it's called *forward pass* (i.e. we pass the inputs forward in order to get the loss). 

In [ ]:
draw_dot(L)

"Backpropagation is just a recursive application of chain rule backwards through the computation graph".

en la regla de la cadena se entiende que hay que multiplicar porque estamos trabajando con el ratio de una respecto al ratio de otra. Son ratios, joder.

Neural networks, in their simplest case, are called multilayer perceptrons.

¿No es maravilloso que de este aparatus matemático, que trata de imitar cómo funcionan las neuronas, puedan sacarse redes complejas que entiendan, por ejemplo, a detectar números?

Me encantaría volver a trabajar con el dataset de números y ver qué se me ocurre con eso.

I left at 1h mark. Maybe I should try to follow the example of the layer he was explaining.

me he quedado en 01:48:00. Joder se me está haciendo denso. No sé si escribir el código a la que le veo o qué. Sí. Mejor así.

In order to visualize how a **Layer** of a neural network is modelled:

![Layer diagram](../images/neural_net2.jpeg)

So we have the *inputs*, which interact with each and every neuron in each layer, but the neurons within a layer don't interact with each other.

In [69]:
class Neuron:
    def __init__(self, nin):
        # weights
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        # bias
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # this is the raw activation
        activation = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        # and now we have to pass it through our "normalization" (is that called like that?) function
        out = activation.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:  # Multi Layer Perceptron
    def __init__(self, nin: int, nouts: list[int]):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [70]:
mlp = MLP(3, [4, 4, 1])

# x = [2.0, 3.0, -1.0]
# mlp(x)

In [81]:
# for each array of this list, we want the MLP (neural net) to output ys[i] when fed xs[i]
# this could be considered a binary classifier
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # desired targets
ypred = [mlp(x) for x in xs]
ypred

[Value(data=0.7244475552029543, ),
 Value(data=-0.9584193543603463, ),
 Value(data=-0.730795765547471, ),
 Value(data=0.4901686647500112, )]

We are going to define a **loss**, which is a measure of how far our MLP is striving from our desired targets

In [82]:
individual_losses = [(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)]
loss = sum(individual_losses)
print("Individual losses:", individual_losses)
print("Loss:", loss)

Individual losses: [Value(data=0.07592914983362889, ), Value(data=0.0017289500918104515, ), Value(data=0.07247091984717217, ), Value(data=0.25992799040278647, )]
Loss: Value(data=0.410057010175398, )


In [74]:
loss.backward()

In [80]:
mlp.layers[0].neurons[0].w[0].grad

0.08880736726421697

Remember: the `grad` meant how a specific weight (Value) is affecting the end result (in this case, **loss**). So if `grad > 0`, it means that pumping that weight value up will increase the loss, while if `grad < 0`, if we increase that weight the loss will go down.

In [ ]:
for p in mlp.parameters():
    learning_rate = 0.01
    p.data += -learning_rate * p.grad

CONCLUSION: a neural net is just a BIG mathematical expression. We can compute the derivatives (how they affect the output) (basically the chain rule) of each parameter, and "tune" them as desired. 

As long as we know how to `forward` and how to `backward` through a node in our neural net, we can work with it, optimize it, and use it to predict (and train to improve it). Pytorch even allows us to do so.